In [31]:
%pip install pandas
%pip install matplotlib


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from pricer import suggest_price



In [33]:
#load synthetic data
stock = pd.read_csv("data/stock.csv", parse_dates=["purchased_at"])
competitor_prices = pd.read_csv("data/competitor_prices.csv", parse_dates=["timestamp"])
sales_history = pd.read_csv("data/sales_history.csv", parse_dates=["timestamp"])

In [34]:
def baseline_cost_plus(sku, now, competitor_snapshot, unit_cost):
    price = unit_cost * 1.3
    return {"chosen_price": price, "freshness_factor": 1.0, "rationale": "Cost+1.3 baseline"}

def baseline_cheapest_competitor(sku, now, competitor_snapshot, unit_cost):
    price = min(competitor_snapshot) if competitor_snapshot else unit_cost * 1.3
    return {"chosen_price": price, "freshness_factor": 1.0, "rationale": "Match cheapest competitor"}


In [35]:
def run_simulation(stock, competitor_prices, pricer_fn, days=7):
    results = []
    start_time = min(stock["purchased_at"])
    end_time = start_time + timedelta(days=days)
    timeline = pd.date_range(start=start_time, end=end_time, freq="30min")

    inventory = {row["sku_id"]: row["quantity"] for _, row in stock.iterrows()}

    for now in timeline:
        for _, sku in stock.iterrows():
            sku_id = sku["sku_id"]
            if inventory[sku_id] <= 0:
                continue

            comp_prices = competitor_prices.loc[
                (competitor_prices["timestamp"] == now) &
                (competitor_prices["product"] == sku["product"])
            ]["price"].tolist()

            result = pricer_fn(sku, now, comp_prices, sku["unit_cost_xaf"])
            price = result["chosen_price"]

            # Demand curve
            q0 = 10
            a = 0.8
            p_ref = np.mean(comp_prices) if comp_prices else price
            freshness = result["freshness_factor"]

            demand = q0 * np.exp(-a * (price - p_ref) / p_ref) * freshness
            demand = min(demand, inventory[sku_id])

            inventory[sku_id] -= demand

            results.append({
                "time": now,
                "sku": sku_id,
                "product": sku["product"],
                "price": price,
                "sold_qty": demand,
                "remaining_qty": inventory[sku_id],
                "profit": demand * (price - sku["unit_cost_xaf"]),
                "waste": max(0, inventory[sku_id] if now >= sku["purchased_at"] + timedelta(hours=sku["shelf_life_hours"]) else 0),
                "margin": (price - sku["unit_cost_xaf"]) / sku["unit_cost_xaf"]
            })

    return pd.DataFrame(results)


In [36]:
results_pricer = run_simulation(stock, competitor_prices, suggest_price)
results_costplus = run_simulation(stock, competitor_prices, baseline_cost_plus)
results_cheapest = run_simulation(stock, competitor_prices, baseline_cheapest_competitor)


TypeError: '>' not supported between instances of 'complex' and 'float'

In [ ]:
def plot_results(df, label):
    profit = df.groupby("time")["profit"].sum()
    waste = df.groupby("time")["waste"].sum()
    margin = df.groupby("time")["margin"].mean()

    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1)
    profit.plot(title=f"Profit - {label}")
    plt.subplot(1,3,2)
    waste.plot(title=f"Waste (kg) - {label}")
    plt.subplot(1,3,3)
    margin.plot(title=f"Margin - {label}")
    plt.show()

plot_results(results_pricer, "Dynamic Pricer")
plot_results(results_costplus, "Cost+ Baseline")
plot_results(results_cheapest, "Cheapest Competitor")


NameError: name 'results_pricer' is not defined